# DINOv2 Baseline Diagnostic & Ablation Study Evaluation

This notebook presents the diagnostic investigations and ablation evaluations for our **DINOv2 Baseline** on the DogReID-1553 benchmark:

1. **Backbone Adaptation Depth (Transformer Layer Unfreezing)**: Evaluating retrieval performance scaling across transformer unfreezing depths (0, 2, 4, 6 (baseline), 8, 10 blocks, and Full Fine-tuning).
2. **Subject Masking & Crop Quality**: Disentangling dog biometric features from background context and evaluating detection crop precision.
3. **Number of Frames (Temporal Scaling)**: Evaluating inference scaling across 1, 4, 8 (baseline), 16, 32, and 64 frames with dynamic temporal attention pooling.
4. **Feature Pooling & Aggregation**: Comparing learnable temporal self-attention against mean/max pooling across frames, as well as spatial token pooling within the backbone.
5. **Gallery Size Sensitivity & Failure Analysis**: Evaluating retrieval robustness as the gallery expands from 1 video per identity (38.9% / 777 videos) to 100% (1,998 videos).

All metrics are reported with **95% Bootstrap Confidence Intervals** (m = 100 iterations).

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add root directory to sys.path
CURRENT_DIR = Path.cwd()
ROOT_DIR = CURRENT_DIR if (CURRENT_DIR / "evaluation").exists() else CURRENT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))
if str(ROOT_DIR / "evaluation") not in sys.path:
    sys.path.append(str(ROOT_DIR / "evaluation"))

from evaluation.evaluation_utils import bootstrap_from_csv
print("Loaded evaluation_utils successfully!")
print(f"Project root: {ROOT_DIR}")

Loaded evaluation_utils successfully!
Project root: /d/hpc/projects/FRI/mm12755/DogReID-1553/DogReID-1553


## 1. Backbone Adaptation Depth Scaling (Transformer Layer Unfreezing)

Evaluation of retrieval accuracy across varying adaptation depths of the DINOv2 ViT-B/14 backbone:
- **0 Blocks (Frozen Backbone)**: Only temporal attention pooling and BNNeck head trained; backbone weights entirely frozen.
- **2 Blocks**: Blocks 10–11 unfrozen.
- **4 Blocks**: Blocks 8–11 unfrozen.
- **6 Blocks (Baseline)**: Blocks 6–11 unfrozen (chosen baseline configuration).
- **8 Blocks**: Blocks 4–11 unfrozen.
- **10 Blocks**: Blocks 2–11 unfrozen.
- **Full Fine-tune**: All 12 transformer blocks, patch embed, and LayerNorms unfrozen.

In [ ]:
# =====================================================================
# 1. Backbone Adaptation Depth Scaling Table
# =====================================================================
ABLATIONS_DIR = ROOT_DIR / "evaluation" / "csvs_ablations"
CLOSED_DIR = ROOT_DIR / "evaluation" / "csvs_closed_results"

depth_stages = [
    {
        "key": "unfreeze0",
        "display_name": "Frozen Backbone (0 blocks)",
        "blocks": 0,
        "label": "0\n(Frozen)",
        "short": "0",
        "path": ABLATIONS_DIR / "unfreeze0_vid2vid" / "closed_dist_matrix.csv",
        "desc": "Only temporal pooling + BNNeck trained",
    },
    {
        "key": "unfreeze2",
        "display_name": "Unfreeze 2 blocks",
        "blocks": 2,
        "label": "2",
        "short": "2",
        "path": ABLATIONS_DIR / "unfreeze2_vid2vid" / "closed_dist_matrix.csv",
        "desc": "Blocks 10-11 unfrozen",
    },
    {
        "key": "unfreeze4",
        "display_name": "Unfreeze 4 blocks",
        "blocks": 4,
        "label": "4",
        "short": "4",
        "path": ABLATIONS_DIR / "unfreeze4_vid2vid" / "closed_dist_matrix.csv",
        "desc": "Blocks 8-11 unfrozen",
    },
    {
        "key": "baseline",
        "display_name": "Unfreeze 6 blocks (Baseline)",
        "blocks": 6,
        "label": "6\n(Baseline)",
        "short": "6",
        "is_base": True,
        "path": CLOSED_DIR / "baseline_vid2vid" / "closed_dist_matrix.csv",
        "fallback_path": ABLATIONS_DIR / "baseline_vid2vid" / "closed_dist_matrix.csv",
        "desc": "Blocks 6-11 unfrozen (Chosen baseline)",
    },
    {
        "key": "unfreeze8",
        "display_name": "Unfreeze 8 blocks",
        "blocks": 8,
        "label": "8",
        "short": "8",
        "path": ABLATIONS_DIR / "unfreeze8_vid2vid" / "closed_dist_matrix.csv",
        "desc": "Blocks 4-11 unfrozen",
    },
    {
        "key": "unfreeze10",
        "display_name": "Unfreeze 10 blocks",
        "blocks": 10,
        "label": "10",
        "short": "10",
        "path": ABLATIONS_DIR / "unfreeze10_vid2vid" / "closed_dist_matrix.csv",
        "desc": "Blocks 2-11 unfrozen",
    },
    {
        "key": "full_finetune",
        "display_name": "Full Fine-tune",
        "blocks": 12,
        "label": "12\n(Full)",
        "short": "12 (Full)",
        "path": ABLATIONS_DIR / "full_finetune_vid2vid" / "closed_dist_matrix.csv",
        "desc": "All 12 blocks + patch embed + norm unfrozen",
    },
]

# Baseline reference metrics for deltas
base_entry = next(s for s in depth_stages if s.get("is_base"))
base_path = base_entry["path"] if base_entry["path"].exists() else base_entry.get("fallback_path")
base_res = bootstrap_from_csv(str(base_path), m=100, mode="closed")
base_r1 = base_res["cmc_boot_mean"][0] * 100
base_map = base_res["mAP_boot_mean"] * 100

depth_results = {}
depth_rows = []

for s in depth_stages:
    p = s["path"] if s["path"].exists() and s["path"].stat().st_size > 0 else (s.get("fallback_path") if s.get("fallback_path") and s["fallback_path"].exists() and s["fallback_path"].stat().st_size > 0 else None)
    if not p:
        depth_rows.append({
            "Configuration": s["display_name"],
            "Unfrozen Blocks": s["short"],
            "Rank-1 (%) [95% CI]": "Pending...",
            "Rank-5 (%) [95% CI]": "Pending...",
            "mAP (%) [95% CI]": "Pending...",
            "Δ (R-1 / mAP)": "-",
            "Status": "In Progress",
            "Description": s["desc"],
        })
        continue

    res = bootstrap_from_csv(str(p), m=100, mode="closed")
    m_mean = res.get("mAP_boot_mean", 0.0) * 100
    m_std = res.get("mAP_std", 0.0) * 100
    m_ci = res.get("mAP_ci", (0.0, 0.0))
    m_l, m_u = m_ci[0] * 100, m_ci[1] * 100

    cmc_boot = res.get("cmc_boot_mean", [0.0]*10)
    cmc_std = res.get("cmc_std", np.zeros(10))
    cmc_l = res.get("cmc_ci_lower", cmc_boot)
    cmc_u = res.get("cmc_ci_upper", cmc_boot)

    r1 = cmc_boot[0] * 100
    r1_std = cmc_std[0] * 100 if len(cmc_std) > 0 else 0.0
    r1_l, r1_u = cmc_l[0] * 100, cmc_u[0] * 100

    r5 = cmc_boot[4] * 100 if len(cmc_boot) >= 5 else 0.0
    r5_std = cmc_std[4] * 100 if len(cmc_std) >= 5 else 0.0
    r5_l, r5_u = cmc_l[4] * 100, cmc_u[4] * 100

    delta = "Reference" if s.get("is_base") else f"{(r1 - base_r1):+.1f} / {(m_mean - base_map):+.1f}"

    depth_results[s["key"]] = {
        "stage": s,
        "r1": r1, "r1_l": r1_l, "r1_u": r1_u,
        "map": m_mean, "map_l": m_l, "map_u": m_u,
    }

    depth_rows.append({
        "Configuration": s["display_name"],
        "Unfrozen Blocks": s["short"],
        "Rank-1 (%) [95% CI]": f"{r1:.1f} ± {r1_std:.1f} [{r1_l:.1f}-{r1_u:.1f}]",
        "Rank-5 (%) [95% CI]": f"{r5:.1f} ± {r5_std:.1f} [{r5_l:.1f}-{r5_u:.1f}]",
        "mAP (%) [95% CI]": f"{m_mean:.1f} ± {m_std:.1f} [{m_l:.1f}-{m_u:.1f}]",
        "Δ (R-1 / mAP)": delta,
        "Status": "Verified",
        "Description": s["desc"],
    })

df_depth = pd.DataFrame(depth_rows)
print("=== Backbone Adaptation Depth Scaling (Rank-1 & mAP with 95% Confidence Intervals) ===")
display(df_depth)


-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
=== Backbone Adaptation 

### Backbone Adaptation Depth Scaling Curve (Rank-1 & mAP with 95% Confidence Intervals)

This plot visualizes how retrieval performance scales as transformer blocks are progressively unfrozen,
from completely frozen (0 blocks) up to full backbone fine-tuning, including 95% bootstrap confidence intervals.


In [ ]:
# =====================================================================
# Plot: Performance vs. Backbone Adaptation Depth (with 95% Bootstrap CIs)
# =====================================================================
plot_stages = []
for s in depth_stages:
    k = s["key"]
    if k in depth_results:
        st = depth_results[k]
        plot_stages.append({
            "key": k,
            "label": s["label"],
            "short": s["short"],
            "is_base": s.get("is_base", False),
            "r1": st["r1"],
            "r1_l": st["r1_l"],
            "r1_u": st["r1_u"],
            "map": st["map"],
            "map_l": st["map_l"],
            "map_u": st["map_u"],
        })

if len(plot_stages) >= 2:
    fig, ax = plt.subplots(figsize=(11.5, 7.0), dpi=200)
    x_indices = np.arange(len(plot_stages))
    labels = [s["label"] for s in plot_stages]

    r1_vals = np.array([s["r1"] for s in plot_stages])
    r1_lows = np.array([s["r1_l"] for s in plot_stages])
    r1_highs = np.array([s["r1_u"] for s in plot_stages])

    map_vals = np.array([s["map"] for s in plot_stages])
    map_lows = np.array([s["map_l"] for s in plot_stages])
    map_highs = np.array([s["map_u"] for s in plot_stages])

    # Shaded 95% Confidence Interval bands
    ax.fill_between(x_indices, r1_lows, r1_highs, color="#1f77b4", alpha=0.15, label="_nolegend_")
    ax.fill_between(x_indices, map_lows, map_highs, color="#d95f02", alpha=0.15, label="_nolegend_")

    # Curves with error bars
    ax.errorbar(x_indices, r1_vals, yerr=[r1_vals - r1_lows, r1_highs - r1_vals],
                fmt="-o", color="#1f77b4", label="Rank-1 (%)",
                linewidth=2.8, markersize=8.5, capsize=5.5, capthick=1.8)

    ax.errorbar(x_indices, map_vals, yerr=[map_vals - map_lows, map_highs - map_vals],
                fmt="-s", color="#d95f02", label="mAP (%)",
                linewidth=2.8, markersize=8.5, capsize=5.5, capthick=1.8)

    # Annotate values just outside the confidence intervals
    for i, s in enumerate(plot_stages):
        ax.annotate(
            f"{s['r1']:.1f}%",
            xy=(i, s["r1_u"]),
            xytext=(-6, 5),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=15,
            color="#1f77b4",
            weight="bold",
        )

        ax.annotate(
            f"{s['map']:.1f}%",
            xy=(i, s["map_l"]),
            xytext=(8, -5),
            textcoords="offset points",
            ha="center",
            va="top",
            fontsize=15,
            color="#b84d00",
            weight="bold",
        )

    ax.set_xticks(x_indices)
    ax.set_xticklabels(labels, fontsize=16)
    ax.tick_params(axis="y", labelsize=15)
    ax.set_xlabel("Backbone Adaptation Depth (Unfrozen Blocks)", fontsize=18, labelpad=10)
    ax.set_ylabel("Score (%)", fontsize=18, labelpad=10)
    ax.grid(True, linestyle="--", alpha=0.4)

    # Dynamic y-limit with breathing room
    y_min = max(0.0, min(map_lows) - 3.0)
    y_max = min(100.0, max(r1_highs) + 4.0)
    ax.set_ylim(y_min, y_max)

    ax.legend(loc="lower right", framealpha=0.92, fontsize=18)
    plt.tight_layout()

    # Save publication figure
    fig_dir = ROOT_DIR / "evaluation" / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(fig_dir / "unfreezing_stages_scaling.png", dpi=300, bbox_inches="tight")
    plt.savefig(fig_dir / "unfreezing_stages_scaling.pdf", bbox_inches="tight")
    print("-> Saved figure to " + str(fig_dir / "unfreezing_stages_scaling.png"))
    plt.show()
else:
    print("[INFO] Insufficient depth stages discovered yet to render curve plot.")


-> Saved figure to /d/hpc/projects/FRI/mm12755/DogReID-1553/DogReID-1553/evaluation/figures/unfreezing_stages_scaling.png


## 2. Subject Masking & Crop Quality Diagnostic Evaluation

Diagnostic evaluation to determine how much the model relies on **dog appearance vs. background contextual clues**, and the effect of **bounding box localization noise**:
- **V2V YOLO-Masked**: The dog region is masked out (blacked out) across all video frames to evaluate background bias.
- **I2V Standard (YOLO Crop)**: Automated YOLO detection bounding boxes on single image query vs. video gallery.
- **I2V GT Crop**: Precise ground-truth human-annotated bounding box on image query vs. video gallery.
- **I2V Masked Queries**: Subject masked out in query image (YOLO-masked vs. GT-masked).

In [2]:
# =====================================================================
# 2. Subject Masking & Crop Quality Diagnostic Table
# =====================================================================
DIAG_DIR = ROOT_DIR / "evaluation" / "csvs_diagnostics"
CLOSED_DIR = ROOT_DIR / "evaluation" / "csvs_closed_results"
BASE_CKPT = ROOT_DIR / "checkpoints" / "baseline" / "closed_baseline_ep50" / "latest_model.pth"
BASE_MTIME = BASE_CKPT.stat().st_mtime if BASE_CKPT.exists() else 0

def get_row(name, path, fallback_path=None, ref_r1=None, ref_map=None):
    p = path if path.exists() and path.stat().st_size > 0 else (fallback_path if fallback_path and fallback_path.exists() and fallback_path.stat().st_size > 0 else None)
    if not p or p.stat().st_mtime < BASE_MTIME:
        return {"Setting": name, "Rank-1 (%) [95% CI]": "Pending...", "Rank-5 (%) [95% CI]": "Pending...", "mAP (%) [95% CI]": "Pending...", "Δ (R-1 / mAP)": "-", "Status": "In Progress"}
    res = bootstrap_from_csv(str(p), m=100, mode="closed")
    m_mean = res.get("mAP_boot_mean", 0.0) * 100
    m_std = res.get("mAP_std", 0.0) * 100
    m_ci = res.get("mAP_ci", (0.0, 0.0))
    cmc_boot = res.get("cmc_boot_mean", [0.0]*10)
    cmc_std = res.get("cmc_std", np.zeros(10))
    cmc_l = res.get("cmc_ci_lower", cmc_boot)
    cmc_u = res.get("cmc_ci_upper", cmc_boot)
    cmc = cmc_boot[0] * 100
    r1_std = cmc_std[0] * 100 if len(cmc_std) > 0 else 0.0
    cmc_ci = (cmc_l[0] * 100, cmc_u[0] * 100)
    r5 = cmc_boot[4] * 100 if len(cmc_boot) >= 5 else 0.0
    r5_std = cmc_std[4] * 100 if len(cmc_std) >= 5 else 0.0
    r5_ci = (cmc_l[4] * 100, cmc_u[4] * 100) if len(cmc_l) >= 5 else (r5, r5)
    delta = "Reference" if ref_r1 is None else f"{(cmc - ref_r1):+.1f} / {(m_mean - ref_map):+.1f}"
    return {
        "Setting": name,
        "Rank-1 (%) [95% CI]": f"{cmc:.1f} ± {r1_std:.1f} [{cmc_ci[0]:.1f}-{cmc_ci[1]:.1f}]",
        "Rank-5 (%) [95% CI]": f"{r5:.1f} ± {r5_std:.1f} [{r5_ci[0]:.1f}-{r5_ci[1]:.1f}]",
        "mAP (%) [95% CI]": f"{m_mean:.1f} ± {m_std:.1f} [{m_ci[0]*100:.1f}-{m_ci[1]*100:.1f}]",
        "Δ (R-1 / mAP)": delta,
        "Status": "Verified",
        "_r1": cmc, "_r5": r5, "_map": m_mean
    }

base_v2v = get_row("V2V Standard (Unmasked)", CLOSED_DIR / "baseline_vid2vid" / "closed_dist_matrix.csv", ROOT_DIR / "evaluation/csvs_ablations/baseline/closed_dist_matrix.csv")
v2v_r1, v2v_map = base_v2v.get("_r1"), base_v2v.get("_map")

base_i2v = get_row("I2V Standard (YOLO Crop, Unmasked)", DIAG_DIR / "baseline_img2vid_unmasked_yolo" / "closed_dist_matrix.csv", CLOSED_DIR / "baseline_img2vid" / "closed_dist_matrix.csv")
i2v_r1, i2v_map = base_i2v.get("_r1"), base_i2v.get("_map")

masking_rows = [
    base_v2v,
    get_row("V2V YOLO-Masked (Background-Only)", DIAG_DIR / "baseline_vid2vid_masked" / "closed_dist_matrix.csv", ref_r1=v2v_r1, ref_map=v2v_map),
    base_i2v,
    get_row("I2V GT Crop (Unmasked)", DIAG_DIR / "baseline_img2vid_unmasked_gt" / "closed_dist_matrix.csv", ref_r1=i2v_r1, ref_map=i2v_map),
    get_row("I2V YOLO-Masked Query", DIAG_DIR / "baseline_img2vid_masked_yolo" / "closed_dist_matrix.csv", ref_r1=i2v_r1, ref_map=i2v_map),
    get_row("I2V GT-Masked Query", DIAG_DIR / "baseline_img2vid_masked_gt" / "closed_dist_matrix.csv", ref_r1=i2v_r1, ref_map=i2v_map),
]
df_masking = pd.DataFrame(masking_rows)[["Setting", "Rank-1 (%) [95% CI]", "Rank-5 (%) [95% CI]", "mAP (%) [95% CI]", "Δ (R-1 / mAP)", "Status"]]

print("=== Subject Masking & Crop Quality Diagnostic ===")
display(df_masking)


-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
=== Subject Masking & Crop Quality Diagnostic ===
                              Setting  ...    Status
0             V2V Standard (Unmasked)  ...  Verified
1   V2V YOLO-Masked (Background-Only)  ...  Verified
2  I2V Standard (YOLO Crop, Unmasked)  ...  Verified
3     

## 3. Number of Frames (Temporal Frame Scaling)

Evaluation of retrieval accuracy across varying video temporal sampling rates (1, 4, 8, 16, 32, and 64 frames per tracklet clip).
Our baseline model is trained on 8 frames with dynamic self-attention pooling.

In [3]:
# =====================================================================
# 3. Number of Frames (Temporal Scaling) Table
# =====================================================================
f8_r1, f8_map = v2v_r1, v2v_map
frame_rows = [
    get_row("1 Frame", DIAG_DIR / "baseline_vid2vid_len1" / "closed_dist_matrix.csv", ref_r1=f8_r1, ref_map=f8_map),
    get_row("4 Frames", DIAG_DIR / "baseline_vid2vid_len4" / "closed_dist_matrix.csv", ref_r1=f8_r1, ref_map=f8_map),
    {"Setting": "8 Frames", "Rank-1 (%) [95% CI]": base_v2v["Rank-1 (%) [95% CI]"], "Rank-5 (%) [95% CI]": base_v2v["Rank-5 (%) [95% CI]"], "mAP (%) [95% CI]": base_v2v["mAP (%) [95% CI]"], "Δ (R-1 / mAP)": "Reference", "Status": "Verified", "_r1": f8_r1, "_r5": base_v2v.get("_r5"), "_map": f8_map},
    get_row("16 Frames", DIAG_DIR / "baseline_vid2vid_len16" / "closed_dist_matrix.csv", ref_r1=f8_r1, ref_map=f8_map),
    get_row("32 Frames", DIAG_DIR / "baseline_vid2vid_len32" / "closed_dist_matrix.csv", ref_r1=f8_r1, ref_map=f8_map),
    get_row("64 Frames", DIAG_DIR / "baseline_vid2vid_len64" / "closed_dist_matrix.csv", ref_r1=f8_r1, ref_map=f8_map),
]
df_frames = pd.DataFrame(frame_rows)[["Setting", "Rank-1 (%) [95% CI]", "Rank-5 (%) [95% CI]", "mAP (%) [95% CI]", "Δ (R-1 / mAP)", "Status"]]

print("=== Number of Frames Inference Diagnostic ===")
display(df_frames)


-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
=== Number of Frames Inference Diagnostic ===
     Setting     Rank-1 (%) [95% CI]  ...  Δ (R-1 / mAP)    Status
0    1 Frame  43.5 ± 1.3 [41.2-46.1]  ...  -18.7 / -16.3  Verified
1   4 Frames  58.2 ± 1.5 [55.7-61.1]  ...    -4.0 / -2.9  Verified
2   8 Frames  62.2 ± 1.5 [59.4-64.8]  ...      Reference  Verified
3  16 Frames  62.3 ± 1.5 [59.9-66.0]  ...    +0.2 / +1.0  Verified
4  32 Fra

### Temporal Frame Scaling Curve (Rank-1 & mAP with 95% Confidence Intervals)

This plot visualizes how retrieval performance scales as the number of sampled video frames increases from 1 to 64 frames per tracklet clip.

In [4]:
# =====================================================================
# Plot: Performance vs. Number of Frames (with 95% Bootstrap CIs)
# =====================================================================
frame_plot_defs = [
    {"frames": 1, "label": "1", "path": DIAG_DIR / "baseline_vid2vid_len1" / "closed_dist_matrix.csv"},
    {"frames": 4, "label": "4", "path": DIAG_DIR / "baseline_vid2vid_len4" / "closed_dist_matrix.csv"},
    {"frames": 8, "label": "8 (Baseline)", "path": CLOSED_DIR / "baseline_vid2vid" / "closed_dist_matrix.csv"},
    {"frames": 16, "label": "16", "path": DIAG_DIR / "baseline_vid2vid_len16" / "closed_dist_matrix.csv"},
    {"frames": 32, "label": "32", "path": DIAG_DIR / "baseline_vid2vid_len32" / "closed_dist_matrix.csv"},
    {"frames": 64, "label": "64", "path": DIAG_DIR / "baseline_vid2vid_len64" / "closed_dist_matrix.csv"},
]

frames_eval = []
for item in frame_plot_defs:
    p = item["path"]
    if p.exists() and p.stat().st_size > 0 and p.stat().st_mtime >= BASE_MTIME:
        res = bootstrap_from_csv(str(p), m=100, mode="closed")
        r1 = res["cmc_boot_mean"][0] * 100
        r1_l = res["cmc_ci_lower"][0] * 100
        r1_u = res["cmc_ci_upper"][0] * 100
        m_val = res["mAP_boot_mean"] * 100
        m_l = res["mAP_ci"][0] * 100
        m_u = res["mAP_ci"][1] * 100
        frames_eval.append({
            "frames": item["frames"],
            "label": item["label"],
            "r1": r1,
            "r1_err": [[max(0.0, r1 - r1_l)], [max(0.0, r1_u - r1)]],
            "map": m_val,
            "map_err": [[max(0.0, m_val - m_l)], [max(0.0, m_u - m_val)]]
        })

if len(frames_eval) >= 2:
    fig, ax = plt.subplots(figsize=(9, 5.5), dpi=200)
    x = [item["frames"] for item in frames_eval]
    r1_y = [item["r1"] for item in frames_eval]
    r1_errs = np.array([item["r1_err"] for item in frames_eval]).squeeze().T
    if r1_errs.ndim == 1:
        r1_errs = r1_errs.reshape(2, 1)

    map_y = [item["map"] for item in frames_eval]
    map_errs = np.array([item["map_err"] for item in frames_eval]).squeeze().T
    if map_errs.ndim == 1:
        map_errs = map_errs.reshape(2, 1)

    ax.errorbar(x, r1_y, yerr=r1_errs, fmt="-o", color="#1f77b4", lw=2.2, capsize=4, elinewidth=1.5, label="Rank-1 Accuracy (%)")
    ax.errorbar(x, map_y, yerr=map_errs, fmt="-s", color="#ff7f0e", lw=2.2, capsize=4, elinewidth=1.5, label="mAP (%)")

    ax.set_xscale("log", base=2)
    ax.set_xticks(x)
    ax.set_xticklabels([str(val) for val in x], fontsize=11)
    ax.set_xlabel("Number of Sampled Frames per Video Clip (Log2 Scale)", fontsize=12, fontweight="bold")
    ax.set_ylabel("Performance Metric (%)", fontsize=12, fontweight="bold")
    ax.set_title("Temporal Scaling: Identification Accuracy vs. Number of Frames", fontsize=13, fontweight="bold", pad=12)
    ax.grid(True, which="both", ls="--", alpha=0.45)
    ax.legend(frameon=True, fontsize=11, loc="lower right")

    fig_dir = ROOT_DIR / "evaluation" / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(fig_dir / "frame_length_scaling.png", dpi=300, bbox_inches="tight")
    plt.savefig(fig_dir / "frame_length_scaling.pdf", bbox_inches="tight")
    print("-> Saved figure to " + str(fig_dir / "frame_length_scaling.png"))
    plt.show()
else:
    print(f"-> [{len(frames_eval)}/6 frame points verified so far] Temporal scaling curve will be rendered once evaluation jobs complete.")


-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> Saved figure to /d/hpc/projects/FRI/mm12755/DogReID-1553/DogReID-1553/evaluation/figures/frame_length_scaling.png


## 4. Feature Pooling & Aggregation Ablation (Temporal & Spatial)

Comparing aggregation strategies:
1. **Temporal Aggregation**: Learnable temporal self-attention across 8 video frames vs. simple temporal average pooling and element-wise max pooling.
2. **Spatial Token Pooling**: Global class token `[CLS]` vs. spatial patch mean pooling, generalized mean pooling (GeM, p=3.0), and combinations.

In [5]:
# =====================================================================
# 4. Feature Pooling & Aggregation: Temporal & Spatial Tables
# =====================================================================
ABLATIONS_DIR = ROOT_DIR / "evaluation" / "csvs_ablations"
CLOSED_DIR = ROOT_DIR / "evaluation" / "csvs_closed_results"

def get_pooling_row(name, path, fallback_path=None, ref_r1=None, ref_map=None, desc=""):
    p = path if path.exists() and path.stat().st_size > 0 else (fallback_path if fallback_path and fallback_path.exists() and fallback_path.stat().st_size > 0 else None)
    if not p or p.stat().st_mtime < BASE_MTIME:
        return {"Method": name, "Rank-1 (%) [95% CI]": "Pending...", "Rank-5 (%) [95% CI]": "Pending...", "mAP (%) [95% CI]": "Pending...", "Δ (R-1 / mAP)": "-", "Status": "In Progress", "Description": desc}
    res = bootstrap_from_csv(str(p), m=100, mode="closed")
    m_mean = res.get("mAP_boot_mean", 0.0) * 100
    m_std = res.get("mAP_std", 0.0) * 100
    m_ci = res.get("mAP_ci", (0.0, 0.0))
    cmc_boot = res.get("cmc_boot_mean", [0.0]*10)
    cmc_std = res.get("cmc_std", np.zeros(10))
    cmc_l = res.get("cmc_ci_lower", cmc_boot)
    cmc_u = res.get("cmc_ci_upper", cmc_boot)
    cmc = cmc_boot[0] * 100
    r1_std = cmc_std[0] * 100 if len(cmc_std) > 0 else 0.0
    cmc_ci = (cmc_l[0] * 100, cmc_u[0] * 100)
    r5 = cmc_boot[4] * 100 if len(cmc_boot) >= 5 else 0.0
    r5_std = cmc_std[4] * 100 if len(cmc_std) >= 5 else 0.0
    r5_ci = (cmc_l[4] * 100, cmc_u[4] * 100) if len(cmc_l) >= 5 else (r5, r5)
    delta = "Reference" if ref_r1 is None else f"{(cmc - ref_r1):+.1f} / {(m_mean - ref_map):+.1f}"
    return {
        "Method": name,
        "Rank-1 (%) [95% CI]": f"{cmc:.1f} ± {r1_std:.1f} [{cmc_ci[0]:.1f}-{cmc_ci[1]:.1f}]",
        "Rank-5 (%) [95% CI]": f"{r5:.1f} ± {r5_std:.1f} [{r5_ci[0]:.1f}-{r5_ci[1]:.1f}]",
        "mAP (%) [95% CI]": f"{m_mean:.1f} ± {m_std:.1f} [{m_ci[0]*100:.1f}-{m_ci[1]*100:.1f}]",
        "Δ (R-1 / mAP)": delta,
        "Status": "Verified",
        "Description": desc,
        "_r1": cmc, "_r5": r5, "_map": m_mean
    }

# 1. Temporal Pooling Across Frames
base_temp = get_pooling_row(
    "Temporal Attention Pooling (Baseline)",
    CLOSED_DIR / "baseline_vid2vid" / "closed_dist_matrix.csv",
    desc="Learnable temporal self-attention across 8 frames"
)
temp_r1, temp_map = base_temp.get("_r1"), base_temp.get("_map")

temporal_rows = [
    base_temp,
    get_pooling_row(
        "Temporal Mean Pooling",
        ABLATIONS_DIR / "temporal_mean_vid2vid" / "closed_dist_matrix.csv",
        ref_r1=temp_r1, ref_map=temp_map,
        desc="Global average pooling across 8 video frames"
    ),
    get_pooling_row(
        "Temporal Max Pooling",
        ABLATIONS_DIR / "temporal_max_vid2vid" / "closed_dist_matrix.csv",
        ref_r1=temp_r1, ref_map=temp_map,
        desc="Element-wise maximum across 8 video frames"
    )
]
df_temporal = pd.DataFrame(temporal_rows)[["Method", "Rank-1 (%) [95% CI]", "Rank-5 (%) [95% CI]", "mAP (%) [95% CI]", "Δ (R-1 / mAP)", "Status", "Description"]]

print("=== Temporal Aggregation (Across Video Frames) ===")
display(df_temporal)



-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
-> [CACHE HIT] Loaded precomputed closed metrics for closed_dist_matrix.csv (from bootstrap_cache_closed_m100_seed42.pkl)
=== Temporal Aggregation (Across Video Frames) ===
                                  Method  ...                                        Description
0  Temporal Attention Pooling (Baseline)  ...  Learnable temporal self-attention across 8 frames
1                  Temporal Mean Pooling  ...       Global average pooling across 8 video frames
2                   Temporal Max Pooling  ...         Element-wise maximum across 8 video frames

[3 rows x 7 columns]


## 5. Gallery Size Sensitivity & Failure Analysis

> [!NOTE]
> The detailed **Gallery Size Sensitivity Analysis** (both total video volume scaling and distractor identity scaling), along with the **Best- vs. Worst-Performing Identity Analysis** and **Inter-Identity Confusion Analysis**, have been transferred to the dedicated evaluation notebook: [evaluate_identity_and_confusion.ipynb](file:///d/hpc/projects/FRI/mm12755/DogReID-1553/DogReID-1553/evaluation/evaluate_identity_and_confusion.ipynb).
